# Chapter 3: Stateful Graph Workflows (LangGraph-Style Orchestration)

## Multi-Agent Analog EDA — PhD-Level Treatment

---

This notebook develops **stateful graph workflows** for long-horizon analog design automation. We treat an agentic EDA stack as a **controlled dynamical system** over design artifacts: each node is an operator (simulator, extractor, LLM-templated script), and edges encode **dataflow, control, and recovery**.

> **Scope note.** To keep the environment **self-contained** (no external services, no API keys), we implement a **minimal in-process graph engine** whose API intentionally mirrors common LangGraph patterns: compiled graphs, **checkpointing**, **conditional routing**, **channel reducers**, and **human-in-the-loop** interrupts. The *concepts* align with LangGraph; the code is small enough to audit in a single sitting.

### Learning objectives

1. Formalize workflows as **stateful labeled graphs** with measurable routing guards.
2. Implement **durable checkpointing** with SQLite for **multi-hour / multi-day** P&R and signoff.
3. Encode **PPA-gated conditional routing** between *resize* loops and *layout* commit.
4. Use **time-travel debugging**: replay to any checkpoint and inspect decision-relevant statistics.
5. Compose a **full analog flow graph** with **parallel precheck branches** merged via reducers.
6. Integrate **approval gates** as stable quiescent states (interrupt / resume).

---

## 3.1 LangGraph fundamentals (conceptual isomorphism)

**LangGraph** models an agent workflow as a **finite controlled graph** whose vertices are computation units (tools, simulators, routers) and whose edges carry **typed, evolving state**.

### 3.1.1 Graph-theoretic skeleton

Let the control structure be a directed graph
\[
G = (V,E), \quad E \subseteq V \times V.
\]
EDA workflows are often **not** DAGs: *resize-until-spec* introduces directed cycles.

### 3.1.2 Stateful transition system

Let the global state live in a product space
\[
x \in \mathcal{X} = \bigtimes_{\ell=1}^{L} \mathcal{X}_{\ell},
\]
where each coordinate is a **channel** (netlist handles, parameters, PPA scalars, logs). Each node $v\in V$ implements an update
\[
F_v : \mathcal{X} \to \mathcal{X},
\]
realized in software as a **partial delta** merged into $x$ via channel reducers (§3.5).

### 3.1.3 Conditional edges as guards

A router implements a measurable map
\[
\phi_v : \mathcal{X} \to V,
\]
selecting the successor vertex. In analog flows, $\phi_v$ typically depends on **PPA slack**, **constraint violations**, and **signoff counters**.

### 3.1.4 Observability and checkpoints

An execution trace is $\tau=(v_k,x_k)_{k=0}^{K}$. **Checkpointing** persists a lossless (or auditable-lossy) encoding of $(x_k,v_k)$ so that we can **resume** after preemption and **replay** for debugging.

---

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('dark_background')

fig, ax = plt.subplots(figsize=(11, 6), facecolor='#0d1117')
ax.set_facecolor('#0d1117')
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis('off')
ax.set_title(
    'Stateful EDA graph: schematic → HITL → sizing → (DRC ∥ LVS) → PPA → layout / resize',
    color='w',
    fontsize=12,
)

nodes = {
    'schematic': (1.0, 3.0),
    'HITL': (2.5, 3.0),
    'sizing': (4.0, 3.0),
    'DRC': (4.0, 4.7),
    'LVS': (4.0, 1.3),
    'PPA': (5.6, 3.0),
    'layout': (8.0, 4.1),
    'resize': (8.0, 1.9),
}

for name, (x, y) in nodes.items():
    ax.add_patch(plt.Circle((x, y), 0.34, color='#58a6ff', alpha=0.22, ec='#58a6ff', lw=2))
    ax.text(x, y, name, ha='center', va='center', fontsize=9, color='#e6edf3')

edges = [
    ('schematic', 'HITL'),
    ('HITL', 'sizing'),
    ('sizing', 'DRC'),
    ('sizing', 'LVS'),
    ('DRC', 'PPA'),
    ('LVS', 'PPA'),
    ('PPA', 'layout'),
    ('PPA', 'resize'),
    ('resize', 'sizing'),
]

for a, b in edges:
    x1, y1 = nodes[a]
    x2, y2 = nodes[b]
    ax.annotate(
        '',
        xy=(x2, y2),
        xytext=(x1, y1),
        arrowprops=dict(arrowstyle='->', color='#8b949e', lw=1.6, connectionstyle='arc3,rad=0.05'),
    )

ax.text(5.6, 2.15, r'$\phi_{\mathrm{PPA}}(x)$', color='#ffa657', fontsize=12)
ax.text(3.35, 3.55, 'parallel', color='#79c0ff', fontsize=9)
ax.text(3.35, 3.35, 'prechecks', color='#79c0ff', fontsize=9)

plt.tight_layout()
plt.show()

## 3.2 Persistent checkpointing for long-running EDA

Place-and-route, RC extraction, and Monte-Carlo verification routinely exceed **many hours** of wall time. Orchestration must support:

1. **Crash recovery** after scheduler preemption.
2. **Deterministic replay** for post-mortems (tool versions, corners, random seeds).
3. **Branching** when engineers explore alternate approvals or constraint relaxations.

### 3.2.1 Checkpoint records

Let $\mathrm{Serialize}$ denote a JSON-safe encoding of channel values (plus pointers to large artifacts on disk). A checkpoint is
\[
c_k = \big(\texttt{thread\_id}, \texttt{checkpoint\_id}, \texttt{parent\_id}, \mathrm{Serialize}(x_k), \eta_k\big),
\]
where $\eta_k$ stores the **active node**, seeds, and semantic tags (e.g., `interrupt='hitl'`).

Parent pointers yield a **tree**; time-travel is tree navigation.

### 3.2.2 `SqliteSaver` (educational, on-disk capable)

Below: a toy `SqliteSaver` with the same *logical* responsibilities as LangGraph’s SQLite checkpointer: **put**, **get latest**, and **list history** per `thread_id`.

For multi-hour P&R, pass a filesystem path (e.g. `SqliteSaver("/path/run.db")`) instead of `:memory:` so checkpoints survive process restarts.

---

In [ ]:
import json
import sqlite3
import uuid
from dataclasses import dataclass
from typing import Any, Dict, List, Optional


@dataclass
class Checkpoint:
    thread_id: str
    checkpoint_id: str
    parent_id: Optional[str]
    channel_values: Dict[str, Any]
    metadata: Dict[str, Any]


class SqliteSaver:
    """Minimal SQLite checkpointer (LangGraph-style semantics, no external deps)."""

    def __init__(self, path: str = ':memory:'):
        self.path = path
        self.conn = sqlite3.connect(path, check_same_thread=False)
        ddl = (
            'CREATE TABLE IF NOT EXISTS checkpoints ('
            'thread_id TEXT NOT NULL,'
            'checkpoint_id TEXT NOT NULL,'
            'parent_id TEXT,'
            'state_json TEXT NOT NULL,'
            'meta_json TEXT NOT NULL,'
            'created REAL DEFAULT (strftime(\'%s\',\'now\')),' 
            'PRIMARY KEY (thread_id, checkpoint_id))'
        )
        self.conn.execute(ddl)
        self.conn.commit()

    def put_checkpoint(self, cp: Checkpoint) -> None:
        self.conn.execute(
            'INSERT OR REPLACE INTO checkpoints VALUES (?,?,?,?,?,strftime(\'%s\',\'now\'))',
            (
                cp.thread_id,
                cp.checkpoint_id,
                cp.parent_id,
                json.dumps(cp.channel_values, default=str),
                json.dumps(cp.metadata, default=str),
            ),
        )
        self.conn.commit()

    def get_latest(self, thread_id: str) -> Optional[Checkpoint]:
        cur = self.conn.execute(
            'SELECT checkpoint_id, parent_id, state_json, meta_json FROM checkpoints '
            'WHERE thread_id = ? ORDER BY rowid DESC LIMIT 1',
            (thread_id,),
        )
        row = cur.fetchone()
        if row is None:
            return None
        cid, pid, sj, mj = row
        return Checkpoint(thread_id, cid, pid, json.loads(sj), json.loads(mj))

    def list_history(self, thread_id: str) -> List[Checkpoint]:
        cur = self.conn.execute(
            'SELECT checkpoint_id, parent_id, state_json, meta_json FROM checkpoints '
            'WHERE thread_id = ? ORDER BY rowid ASC',
            (thread_id,),
        )
        out: List[Checkpoint] = []
        for cid, pid, sj, mj in cur.fetchall():
            out.append(Checkpoint(thread_id, cid, pid, json.loads(sj), json.loads(mj)))
        return out


saver = SqliteSaver(':memory:')
tid = 'analog_block_LNA_v0'
root_id = str(uuid.uuid4())
saver.put_checkpoint(
    Checkpoint(
        tid,
        root_id,
        None,
        {'stage': 'schematic', 'netlist_rev': 1},
        {'node': 'schematic_ingest', 'tool': 'spectre_stub'},
    )
)
child_id = str(uuid.uuid4())
saver.put_checkpoint(
    Checkpoint(
        tid,
        child_id,
        root_id,
        {'stage': 'sizing', 'gm': 12.4e-3, 'id_sat': 1.1e-4},
        {'node': 'sizing_opt', 'iter': 3},
    )
)

latest = saver.get_latest(tid)
print('Latest node:', latest.metadata['node'], '| gm =', latest.channel_values['gm'])
print('History length:', len(saver.list_history(tid)))

## 3.3 Conditional routing: PPA slack → layout vs resize loop

Let $p \in \mathbb{R}^d$ denote PPA sufficient statistics extracted from simulation / surrogate models. Feasibility is often expressed as inequalities $g_i(p) \le 0$ after margining.

Define
\[
\Psi(p) = \sum_{i=1}^{m} \max\{0, g_i(p)\}
\]
as a **soft violation metric** (zero iff all constraints satisfied in the hard sense). A discrete router is
\[
\phi_{\mathrm{PPA}}(x) =
\begin{cases}
\texttt{layout\_commit}, & \Psi(p(x)) = 0,\\
\texttt{resize\_feedback}, & \Psi(p(x)) > 0.
\end{cases}
\]

The next cell uses a **mock surrogate** (no SPICE) and visualizes a feasible region in a 2-D projection.

---

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio

# Inline rendering in Jupyter; SVG fallback avoids spawning a GUI browser in headless runs.
if 'plotly_mimetype+notebook' in pio.renderers:
    pio.renderers.default = 'plotly_mimetype+notebook'
else:
    pio.renderers.default = 'svg'

rng = np.random.default_rng(7)


def ppa_metrics_from_state(gm: float, id_sat: float) -> dict:
    power_uw = 120 + 40 * (gm / 20e-3) ** 2 + rng.normal(0, 3)
    fom_mhz = 2.1e9 * (id_sat / 150e-6) ** 0.35 + rng.normal(0, 5e6)
    return {'power_uw': float(power_uw), 'fom_mhz': float(fom_mhz)}


def routing_decision(metrics: dict) -> str:
    ok_power = metrics['power_uw'] < 190
    ok_fom = metrics['fom_mhz'] > 1.6e9
    return 'layout_commit' if (ok_power and ok_fom) else 'resize_feedback'


grid_gm = np.linspace(6e-3, 22e-3, 55)
grid_id = np.linspace(40e-6, 200e-6, 55)
GM, ID = np.meshgrid(grid_gm, grid_id)
Z = np.zeros_like(GM)
for i in range(GM.shape[0]):
    for j in range(GM.shape[1]):
        m = ppa_metrics_from_state(float(GM[i, j]), float(ID[i, j]))
        Z[i, j] = 1.0 if routing_decision(m) == 'layout_commit' else 0.0

fig = go.Figure(
    data=go.Contour(
        z=Z,
        x=grid_gm * 1e3,
        y=grid_id * 1e6,
        colorscale='Viridis',
        contours=dict(showlines=False),
        colorbar=dict(title='1→layout'),
    )
)
fig.update_layout(
    template='plotly_dark',
    paper_bgcolor='#0d1117',
    plot_bgcolor='#0d1117',
    title='Mock feasibility map: router label in (gm [mS], Id,sat [µA]) projection',
    xaxis_title='gm (mS)',
    yaxis_title='Id,sat (µA)',
)
fig.show()

probe = ppa_metrics_from_state(14e-3, 110e-6)
print('Probe metrics:', probe)
print('Route:', routing_decision(probe))

## 3.4 Time-travel debugging

Let $\mathcal{H} = (c_k)_{k=0}^{K}$ be checkpoint records for a thread. **Time-travel** to index $k^\star$ is the reconstruction map
\[
\mathrm{TT}_{k^\star} : \mathcal{H} \to \mathcal{X}, \quad \mathrm{TT}_{k^\star}(\mathcal{H}) = \mathrm{Deserialize}(\texttt{state\_json}_{k^\star}).
\]

**Causal queries** in EDA debug typically compare $\mathrm{TT}_{k^\star}(\mathcal{H})$ across two branches that diverged after a layout ECO: did mismatch arise from **extraction revision**, **corner**, or **device parameter drift**?

---

In [ ]:
from typing import Tuple


def reconstruct_at(history: List[Checkpoint], index: int) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    if index < 0 or index >= len(history):
        raise IndexError('checkpoint index out of range')
    cp = history[index]
    return cp.channel_values, cp.metadata


saver2 = SqliteSaver(':memory:')
thread = 'demo_thread'
parent = None
for k, stage in enumerate(['schematic', 'sizing', 'ppa', 'layout']):
    cid = str(uuid.uuid4())
    x = {'stage': stage, 'iter': k, 'artifact': f'{stage}_v{k}.json'}
    if stage == 'ppa':
        x['ppa'] = ppa_metrics_from_state(13e-3, 95e-6)
    meta = {'node': f'{stage}_node', 'rng_seed': 1234 + k}
    saver2.put_checkpoint(Checkpoint(thread, cid, parent, x, meta))
    parent = cid

hist = saver2.list_history(thread)
for i in range(len(hist)):
    xv, mv = reconstruct_at(hist, i)
    print(f'k={i}: node={mv["node"]!s:14} stage={xv.get("stage")!s:10} ppa={xv.get("ppa", "—")}')

k_star = 2
x_star, m_star = reconstruct_at(hist, k_star)
print('\nTime-travel to k* =', k_star, '→', x_star)

## 3.5 State channels and reducers (parallel branches)

Partition $\mathcal{X} = \bigtimes_{\ell} \mathcal{X}_\ell$. Suppose two precheck nodes produce partial updates $\Delta^{(1)}, \Delta^{(2)}$ that write to *distinct or shared* channels.

A **reducer** for channel $\ell$ is a binary operator $\oplus_\ell$ used to fold parallel writes:
\[
x'_\ell = x_\ell \oplus_\ell \Delta^{(1)}_\ell \oplus_\ell \Delta^{(2)}_\ell.
\]

Common choices:

- **Append** for event logs (non-commutative ordering fixed by engine scheduling).
- **Max** for worst-case slack across corners.
- **Last-writer-wins** for scalar knobs when later tools intentionally override earlier estimates.

---

In [ ]:
from typing import Any, Callable, Optional

Reducer = Callable[[Any, Any], Any]


def default_reducer(old: Any, new: Any) -> Any:
    if old is None:
        return new
    if isinstance(old, dict) and isinstance(new, dict):
        merged = dict(old)
        merged.update(new)
        return merged
    return new


def append_reducer(old: Optional[list], new: Optional[list]) -> list:
    old = old or []
    new = new or []
    return old + new


def max_reducer(old: Optional[float], new: Optional[float]) -> float:
    vals = [v for v in (old, new) if v is not None]
    return max(vals) if vals else float('-inf')


class ChannelSpec:
    def __init__(self, reducers: Dict[str, Reducer]):
        self.reducers = reducers

    def merge(self, base: Dict[str, Any], delta: Dict[str, Any]) -> Dict[str, Any]:
        out = dict(base)
        for k, v in delta.items():
            fn = self.reducers.get(k, default_reducer)
            out[k] = fn(out.get(k), v)
        return out


spec = ChannelSpec({'log': append_reducer, 'worst_slack_ps': max_reducer})
base = {'gm': 10e-3, 'log': ['start'], 'worst_slack_ps': None}
d_drc = {'log': ['drc:0 violations'], 'worst_slack_ps': 42.0}
d_lvs = {'log': ['lvs: match'], 'worst_slack_ps': 55.0}
merged = spec.merge(spec.merge(base, d_drc), d_lvs)
print(merged)

## 3.6 Human-in-the-loop: approval as interrupt / resume

Let a gate node $v_{\mathrm{H}}$ be equipped with a predicate $A(x) \in \{\top,\bot\}$ meaning “approval granted”. The guarded update is
\[
F_{v_{\mathrm{H}}}(x; u) =
\begin{cases}
x, & u = \emptyset \land A(x)=\bot \quad\text{(suspend)},\\
x \oplus \Delta_{\mathrm{approve}}(u), & A(x)=\top.
\end{cases}
\]

The orchestrator writes a checkpoint tagged `interrupt='hitl'` at suspension; the continuation supplies human input $u$ (digital signatures, waiver IDs, constraints).

---

In [ ]:
from dataclasses import dataclass


@dataclass
class HITLStatus:
    pending: bool
    payload: Dict[str, Any]


def hitl_node(state: Dict[str, Any], human_input: Optional[Dict[str, Any]]):
    if state.get('hitl_approved', False):
        return {}, None
    if human_input is None:
        return {}, HITLStatus(True, {'reason': 'topology delta exceeds policy'})
    if human_input.get('approve'):
        return {'hitl_approved': True, 'approver': human_input.get('user', 'unknown')}, None
    return {'hitl_rejected': True}, None


s0 = {'hitl_approved': False}
d1, intr = hitl_node(s0, None)
print('Awaiting human:', intr)
d2, intr2 = hitl_node({**s0, **d1}, {'approve': True, 'user': 'reviewer_A'})
print('After decision:', d2, intr2)

## 3.7 Custom graph engine + full analog EDA workflow

We compile a workflow analogous to `StateGraph.compile(...)`:

- **Nodes** are pure Python callables $F_v$ returning deltas.
- **Fan-out / fan-in** implements parallel DRC/LVS-style checks with reducer merges.
- **Conditional routing** after `ppa_eval` chooses `layout_commit` vs `resize_feedback`.
- **HITL** suspends at `human_review` until `human_input` is provided; **resume** uses `__resume__.next_node`.
- **SqliteSaver** logs every node transition for time-travel.

---

In [ ]:
from typing import Callable, Dict, List, Optional

NodeFn = Callable[[Dict[str, Any]], Dict[str, Any]]
END = '__END__'


class StateGraph:
    def __init__(self, schema: ChannelSpec):
        self.schema = schema
        self.nodes: Dict[str, NodeFn] = {}
        self.edges: Dict[str, str] = {}
        self.fanout: Dict[str, List[str]] = {}
        self.conditional: Dict[str, Callable[[Dict[str, Any]], str]] = {}
        self.entry_point: Optional[str] = None

    def add_node(self, name: str, fn: NodeFn) -> None:
        self.nodes[name] = fn

    def set_entry_point(self, name: str) -> None:
        self.entry_point = name

    def add_edge(self, src: str, dst: Optional[str]) -> None:
        self.edges[src] = END if dst is None else dst

    def add_fanout(self, src: str, branches: List[str], join: str) -> None:
        self.fanout[src] = list(branches)
        for b in branches:
            self.edges[b] = join

    def add_conditional_edges(self, src: str, router: Callable[[Dict[str, Any]], str]) -> None:
        self.conditional[src] = router

    def compile(self, checkpointer: Optional[SqliteSaver] = None):
        if self.entry_point is None:
            raise ValueError('entry point not set')
        return CompiledGraph(self, checkpointer)


class CompiledGraph:
    def __init__(self, sg: StateGraph, checkpointer: Optional[SqliteSaver]):
        self.sg = sg
        self.checkpointer = checkpointer

    def _checkpoint(self, thread_id: str, parent_id: Optional[str], state: Dict[str, Any], meta: Dict[str, Any]) -> str:
        if self.checkpointer is None:
            return parent_id or 'noop'
        cid = str(uuid.uuid4())
        safe = json.loads(json.dumps(state, default=str))
        self.checkpointer.put_checkpoint(Checkpoint(thread_id, cid, parent_id, safe, meta))
        return cid

    def invoke(
        self,
        state: Dict[str, Any],
        config: Optional[Dict[str, Any]] = None,
        human_input: Optional[Dict[str, Any]] = None,
    ) -> Dict[str, Any]:
        cfg = config or {}
        thread_id = cfg.get('configurable', {}).get('thread_id', 'default_thread')
        max_steps = int(cfg.get('max_steps', 500))

        x = dict(state)
        resume = x.pop('__resume__', None)
        cur = (resume or {}).get('next_node', self.sg.entry_point)
        parent_cp: Optional[str] = (resume or {}).get('parent_checkpoint_id')

        steps = 0
        interrupt = False

        while cur is not None and cur != END and steps < max_steps:
            steps += 1

            if cur == 'human_review':
                delta, intr = hitl_node(x, human_input)
                if intr is not None:
                    cp = self._checkpoint(thread_id, parent_cp, x, {'node': cur, 'interrupt': 'hitl', 'detail': intr.payload})
                    x['__resume__'] = {'next_node': 'human_review', 'parent_checkpoint_id': cp}
                    x['__debug__'] = {'steps': steps, 'interrupt': True}
                    interrupt = True
                    break
                x = self.sg.schema.merge(x, delta)
                parent_cp = self._checkpoint(thread_id, parent_cp, x, {'node': cur, 'interrupt': None})
                cur = self.sg.edges[cur]
                continue

            fn = self.sg.nodes[cur]
            delta = fn(x)
            x = self.sg.schema.merge(x, delta)
            parent_cp = self._checkpoint(thread_id, parent_cp, x, {'node': cur})

            if cur in self.sg.fanout:
                for br in self.sg.fanout[cur]:
                    bd = self.sg.nodes[br](x)
                    x = self.sg.schema.merge(x, bd)
                    parent_cp = self._checkpoint(thread_id, parent_cp, x, {'node': br, 'parallel_group': cur})
                cur = self.sg.edges[self.sg.fanout[cur][0]]
                continue

            if cur in self.sg.conditional:
                cur = self.sg.conditional[cur](x)
                continue

            cur = self.sg.edges.get(cur, END)

        if not interrupt:
            x['__debug__'] = {'steps': steps, 'interrupt': False}
        return x


def build_analog_flow(saver: Optional[SqliteSaver] = None) -> CompiledGraph:
    schema = ChannelSpec(
        {
            'log': append_reducer,
            'worst_slack_ps': max_reducer,
            'resize_cycles': max_reducer,
        }
    )
    g = StateGraph(schema)

    def schematic_ingest(_x):
        return {'stage': 'schematic', 'log': ['parsed netlist (mock)'], 'devices': 38}

    def human_review_stub(_x):
        return {}

    def sizing_opt(x):
        it = int(x.get('resize_cycles', 0))
        gm = 7.5e-3 + 1.1e-3 * it + float(rng.normal(0, 0.15e-3))
        id_sat = 65e-6 + 9e-6 * it + float(rng.normal(0, 1e-6))
        return {'gm': gm, 'id_sat': id_sat, 'log': [f'sizing (cycle {it})']}

    def drc_preflight(_x):
        v = int(rng.integers(0, 2))
        return {'log': [f'drc: violations={v}'], 'worst_slack_ps': float(rng.normal(52, 3))}

    def lvs_preflight(_x):
        return {'log': ['lvs: topology match (mock)'], 'worst_slack_ps': float(rng.normal(49, 2))}

    def ppa_eval(x):
        m = ppa_metrics_from_state(float(x['gm']), float(x['id_sat']))
        return {'stage': 'ppa', 'ppa': m, 'log': ['ppa evaluated (surrogate)']}

    def layout_commit(_x):
        return {'stage': 'layout', 'log': ['committed DEF/LEF placeholders (mock)']}

    def resize_feedback(x):
        c = int(x.get('resize_cycles', 0)) + 1
        return {'resize_cycles': c, 'log': ['resize: tighten budgets / adjust widths (mock)']}

    for name, fn in [
        ('schematic_ingest', schematic_ingest),
        ('human_review', human_review_stub),
        ('sizing_opt', sizing_opt),
        ('drc_preflight', drc_preflight),
        ('lvs_preflight', lvs_preflight),
        ('ppa_eval', ppa_eval),
        ('layout_commit', layout_commit),
        ('resize_feedback', resize_feedback),
    ]:
        g.add_node(name, fn)

    g.set_entry_point('schematic_ingest')
    g.add_edge('schematic_ingest', 'human_review')
    g.add_edge('human_review', 'sizing_opt')
    g.add_fanout('sizing_opt', ['drc_preflight', 'lvs_preflight'], join='ppa_eval')
    g.add_conditional_edges('ppa_eval', lambda x: routing_decision(x['ppa']))
    g.add_edge('layout_commit', None)
    g.add_edge('resize_feedback', 'sizing_opt')

    return g.compile(checkpointer=saver)


flow_saver = SqliteSaver(':memory:')
cg = build_analog_flow(flow_saver)

x0 = {'hitl_approved': False, 'resize_cycles': 0, 'log': [], 'worst_slack_ps': None}
x1 = cg.invoke(x0, config={'configurable': {'thread_id': 'lna_run_A'}, 'max_steps': 500}, human_input=None)
print('After first slice (expect HITL interrupt):', x1.get('__debug__'), x1.get('__resume__'))

x2 = cg.invoke(x1, config={'configurable': {'thread_id': 'lna_run_A'}}, human_input={'approve': True, 'user': 'analog_lead'})
print('Final stage:', x2.get('stage'), '| resize_cycles:', x2.get('resize_cycles'))
print('PPA:', x2.get('ppa'))
print('Log tail:', x2.get('log', [])[-8:])

hist = flow_saver.list_history('lna_run_A')
print('\nCheckpoint count:', len(hist))
print('Time-travel @ -3:', reconstruct_at(hist, len(hist) - 3)[0].get('stage'))

## 3.8 Interactive trace visualization (matplotlib)

We visualize a scalar projection of the checkpoint sequence—here the **mock power** recorded in `ppa.power_uw` when present—to illustrate how **time** and **graph depth** relate in a logged run.

---

In [ ]:
powers = []
labels = []
for cp in flow_saver.list_history('lna_run_A'):
    node = cp.metadata.get('node', '?')
    labels.append(node)
    ppa = cp.channel_values.get('ppa')
    powers.append(float(ppa['power_uw']) if isinstance(ppa, dict) else float('nan'))

idx = np.arange(len(powers))
fig, ax = plt.subplots(figsize=(10, 4), facecolor='#0d1117')
ax.set_facecolor('#0d1117')
ax.plot(idx, powers, color='#79c0ff', lw=2, marker='o', ms=4)
ax.set_title('Checkpoint-indexed mock power trajectory', color='w')
ax.set_xlabel('checkpoint index k', color='#8b949e')
ax.set_ylabel('power (µW)', color='#8b949e')
ax.grid(True, alpha=0.25)
for t in ax.get_xticklabels() + ax.get_yticklabels():
    t.set_color('#c9d1d9')
plt.tight_layout()
plt.show()

## 3.9 Synthesis

- **Graph = controlled state machine:** nodes are $F_v$; conditional edges are measurable routers $\phi_v$.
- **Checkpointing = semantics for reliability:** SQLite (or server-backed stores) materialize $\mathcal{H}$ for resume and audit.
- **Reducers = parallel semantics:** commutative folds (max, sum) vs ordered logs (append) must match engineering intent.
- **HITL = stable quiescence:** explicit interrupts avoid “hidden blocking” inside opaque agents.
- **Time-travel = causal analysis:** compare $\mathrm{TT}_{k^\star}$ across branches when signoff regressions appear.

### Suggested reading (external)

- LangGraph documentation: *Persistence*, *Interrupts*, *Human-in-the-loop* patterns.
- Ramadge–Wonham supervisory control (discrete-event view of guarded transitions).
- Notebook 02 (Core Agentic Architectures) for comparative orchestration patterns.

---